# Databricks to AzureML model endpoint call

This notebook validates that a Databricks runtime can authenticate to AzureML and call a managed online endpoint.

**Prereqs**
- Install packages if missing: `azure-identity`, `requests`.
- Provide credentials via environment variables or Databricks secrets.

**Required environment variables**
- `AZUREML_ENDPOINT_URL` (example: `https://<endpoint>.<region>.inference.ml.azure.com/score`)

**Managed identity or service principal**
- `AZURE_TENANT_ID` (optional if using managed identity)
- `AZURE_CLIENT_ID` (optional if using managed identity)
- `AZURE_CLIENT_SECRET` (optional if using managed identity)

**Optional**
- `AZUREML_SAMPLE_INPUT_JSON` (JSON payload string; if omitted, a default payload is sent)

If your endpoint expects a different schema, set `AZUREML_SAMPLE_INPUT_JSON` to match your scoring input.

In [ ]:
import json
import os
import requests
import logging

# Latest Azure ML SDK v2
from azure.identity import ClientSecretCredential, DefaultAzureCredential, EnvironmentCredential, ChainedTokenCredential
from azure.core.exceptions import HttpResponseError

# Latest Databricks SDK (databricks-sdk)
try:
    from databricks.sdk import WorkspaceClient
    print("✓ Databricks SDK (databricks-sdk) imported successfully")
except ImportError:
    print("Note: databricks-sdk not installed. Install with: pip install databricks-sdk")
    WorkspaceClient = None

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

endpoint_url = os.getenv("AZUREML_ENDPOINT_URL")
if not endpoint_url:
    raise ValueError("Missing AZUREML_ENDPOINT_URL.")

tenant_id = os.getenv("AZURE_TENANT_ID")
client_id = os.getenv("AZURE_CLIENT_ID")
client_secret = os.getenv("AZURE_CLIENT_SECRET")

# Use ChainedTokenCredential for robust auth (latest pattern)
if all([tenant_id, client_id, client_secret]):
    credential = ChainedTokenCredential(
        EnvironmentCredential(),  # Service principal priority
        ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret),
        DefaultAzureCredential()  # Fallback
    )
    logger.info("Using service principal credentials")
else:
    credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
    logger.info("Using default credentials (managed identity or interactive)")

In [ ]:
# Optional: quick sanity check to confirm the token audience
print("Token acquired for AzureML scope.")